# Projek Probabilitas dan Statistika — Kelompok 7 (3 TI A)
**Analisis Statistik Deskriptif Harga Produk, Ongkos Kirim, Waktu Pengiriman, dan Skor Kepuasan Pelanggan E-Commerce (Studi Kasus: Olist Store)**

Anggota: Gede Abhita Tangkas Aryasuta Gandi (2455301073), Farrell Fathurrahman (2455301067), Alif Fatan Akhlaqi (2455301020)
Sumber data: Olist Brazilian E-Commerce (Kaggle) — https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce/data

Notebook ini terdiri dari **dua tahap** dalam satu alur:

| Tahap | Isi |
|---|---|
| **Tahap 1 — Audit dan pembersihan data** | memuat data, struktur, kamus data, missing value, duplikasi, nilai tidak wajar, outlier, keputusan pembersihan |
| **Tahap 2 — Statistika deskriptif dan visualisasi** | A. Elemen data dan tipe variabel · B. Tabel resume statistika deskriptif · C. Distribusi frekuensi · D. Visualisasi tiap variabel · E. Ringkasan menurut kelompok skor · F. Ringkasan temuan |

**Cara menjalankan:** Runtime → *Run all*, lalu upload `3TIA_07_ECommerceOlist.xlsx` saat diminta. Semua grafik dan tabel tersimpan di folder `hasil/` dan diunduh sebagai `hasil_analisis.zip` di sel terakhir. Hanya memakai numpy, pandas, dan matplotlib (scipy hanya diimpor, tidak dipakai untuk uji statistik).

## 0. Setup

In [ ]:
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from scipy import stats

warnings.filterwarnings('ignore')
np.random.seed(42)                      # supaya hasil simulasi selalu sama

# Warna: biru & oranye untuk perbandingan dua kelompok, abu untuk netral
BIRU, ORANYE, AQUA = '#2a78d6', '#eb6834', '#1baf7a'
INK, MUTED, GRID, NETRAL = '#0b0b0b', '#52514e', '#e5e4df', '#c9c8c2'

plt.rcParams.update({
    'figure.dpi': 110, 'savefig.dpi': 200, 'font.size': 10,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': .8, 'axes.axisbelow': True,
    'axes.edgecolor': '#9b9a94', 'axes.labelcolor': MUTED,
    'xtick.color': MUTED, 'ytick.color': MUTED, 'text.color': INK,
    'axes.titleweight': 'bold', 'axes.titlesize': 11, 'axes.titlelocation': 'left',
    'figure.facecolor': 'white'})

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
pd.set_option('display.max_columns', None)

HASIL = 'hasil'
os.makedirs(HASIL, exist_ok=True)
K = {}                                  # wadah angka kunci untuk laporan

def simpan(fig, nama):
    fig.savefig(f'{HASIL}/{nama}.png', bbox_inches='tight', facecolor='white')
    plt.show()

def fmt_p(p):
    return '< 0,001' if p < 0.001 else f'{p:.4f}'.replace('.', ',')

VAR_X = ['Harga_Produk', 'Ongkos_Kirim', 'Waktu_Pengiriman_Hari']
VAR_NUM = VAR_X + ['Skor_Kepuasan']
print('Siap.')

# TAHAP 1 — Audit dan Pembersihan Data

# TAHAP 1 — Audit dan Pembersihan Data

## 1.1 Memuat data
Jalankan sel ini lalu **upload file `3TIA_07_ECommerceOlist.xlsx`** saat muncul tombol *Choose Files*.
(Kalau dijalankan di komputer lokal, otomatis membaca file di folder yang sama.)

In [ ]:
import os
NAMA_FILE = '3TIA_07_ECommerceOlist.xlsx'

try:
    from google.colab import files
    if not os.path.exists(NAMA_FILE):
        uploaded = files.upload()
        NAMA_FILE = list(uploaded.keys())[0]
except ImportError:
    pass  # bukan di Colab

df_raw = pd.read_excel(NAMA_FILE, sheet_name='Data Utama')
kamus  = pd.read_excel(NAMA_FILE, sheet_name='Kamus Data')

print('File       :', NAMA_FILE)
print('Ukuran data:', df_raw.shape[0], 'baris x', df_raw.shape[1], 'kolom')

## 1.2 Struktur dataset

In [ ]:
df_raw.head(10)

In [ ]:
df_raw.info()

In [ ]:
# Ringkasan tipe data per kolom
pd.DataFrame({'Tipe di Python': df_raw.dtypes.astype(str),
              'Jumlah nilai unik': df_raw.nunique()})

## 1.3 Kamus data
Pengecekan *kejelasan variabel, tipe, dan skala* (syarat dari dosen).

In [ ]:
kamus

## 1.4 Pengecekan missing value

In [ ]:
miss = pd.DataFrame({'Jumlah kosong': df_raw.isna().sum(),
                     'Persen (%)': df_raw.isna().mean() * 100})
miss

In [ ]:
# Tipe numerik harus benar-benar numerik (bukan teks)
for c in VAR_NUM:
    print(f'{c:25s} numerik? ->', pd.api.types.is_numeric_dtype(df_raw[c]))

## 1.5 Duplikasi dan unit observasi
Kolom `ID_Pesanan` ternyata bisa muncul lebih dari sekali. Penyebabnya satu pesanan dapat berisi beberapa item produk.
Jadi yang perlu dicek: apakah ada **baris yang sama persis** (duplikat murni), dan berapa pesanan yang punya lebih dari satu baris.

In [ ]:
n_baris        = len(df_raw)
n_pesanan_unik = df_raw['ID_Pesanan'].nunique()
n_dup_id       = df_raw['ID_Pesanan'].duplicated().sum()
n_dup_penuh    = df_raw.duplicated().sum()

print(f'Jumlah baris (item pesanan)        : {n_baris:,}')
print(f'Jumlah ID_Pesanan unik             : {n_pesanan_unik:,}')
print(f'Baris dengan ID_Pesanan berulang   : {n_dup_id:,}')
print(f'Baris duplikat murni (semua kolom) : {n_dup_penuh:,}')

In [ ]:
# Contoh pesanan yang punya lebih dari satu baris
id_ganda = df_raw[df_raw['ID_Pesanan'].duplicated(keep=False)].sort_values('ID_Pesanan')
print('Jumlah baris yang terlibat:', len(id_ganda))
id_ganda.head(8)

In [ ]:
# Pada pesanan multi-item, apakah Ongkos_Kirim / Waktu / Skor sama untuk semua barisnya?
g = id_ganda.groupby('ID_Pesanan').agg(
    n_item=('ID_Pesanan', 'size'),
    ongkir_sama=('Ongkos_Kirim', 'nunique'),
    waktu_sama=('Waktu_Pengiriman_Hari', 'nunique'),
    skor_sama=('Skor_Kepuasan', 'nunique'))
print('Pesanan multi-item                    :', len(g))
print('  - Waktu pengiriman identik          :', (g.waktu_sama == 1).sum())
print('  - Skor kepuasan identik             :', (g.skor_sama == 1).sum())
print('  - Ongkos kirim identik              :', (g.ongkir_sama == 1).sum())

**Catatan interpretasi:** waktu pengiriman selalu identik pada pesanan multi-item, dan skor ulasan identik pada sebagian besar pesanan (keduanya melekat pada *pesanan*), sedangkan ongkos kirim bervariasi per item. Nilai yang terulang itu membuat baris-baris tersebut tidak sepenuhnya independen. Karena itu **unit observasi dalam projek ini kita tetapkan sebagai item pesanan**, dan hal ini dicantumkan sebagai keterbatasan. Pada bagian deskriptif, hasil juga dapat dibaca dengan memperhatikan keterbatasan ini.

## 1.6 Pengecekan nilai tidak wajar

In [ ]:
# Statistik ringkas untuk melihat rentang nilai
df_raw[VAR_NUM].describe().T

In [ ]:
cek = pd.DataFrame({
    'Harga_Produk <= 0'            : [(df_raw['Harga_Produk'] <= 0).sum()],
    'Ongkos_Kirim < 0'             : [(df_raw['Ongkos_Kirim'] < 0).sum()],
    'Ongkos_Kirim = 0 (gratis)'    : [(df_raw['Ongkos_Kirim'] == 0).sum()],
    'Waktu_Pengiriman < 0'         : [(df_raw['Waktu_Pengiriman_Hari'] < 0).sum()],
    'Waktu_Pengiriman = 0'         : [(df_raw['Waktu_Pengiriman_Hari'] == 0).sum()],
    'Skor di luar 1-5'             : [(~df_raw['Skor_Kepuasan'].isin([1,2,3,4,5])).sum()],
    'Skor tidak bulat'             : [(df_raw['Skor_Kepuasan'] % 1 != 0).sum()],
}).T.rename(columns={0: 'Jumlah baris'})
cek

In [ ]:
# Lihat baris dengan waktu pengiriman = 0 hari
df_raw[df_raw['Waktu_Pengiriman_Hari'] == 0]

In [ ]:
# Distribusi skor kepuasan
dist_skor = df_raw['Skor_Kepuasan'].value_counts().sort_index().to_frame('Frekuensi')
dist_skor['Persen (%)'] = dist_skor['Frekuensi'] / len(df_raw) * 100
dist_skor

## 1.7 Audit outlier (hanya dihitung, belum dihapus)
Metode IQR: nilai di luar Q1 − 1,5·IQR atau Q3 + 1,5·IQR.

In [ ]:
def audit_iqr(s):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    bawah, atas = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = ((s < bawah) | (s > atas)).sum()
    return pd.Series({'Q1': q1, 'Q3': q3, 'IQR': iqr, 'Batas bawah': bawah,
                      'Batas atas': atas, 'Jumlah outlier': n_out,
                      'Persen outlier (%)': n_out / len(s) * 100,
                      'Min': s.min(), 'Maks': s.max()})

audit_outlier = df_raw[VAR_NUM[:3]].apply(audit_iqr).T
audit_outlier

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
for a, c in zip(ax, VAR_NUM[:3]):
    a.boxplot(df_raw[c], vert=False, flierprops={'markersize': 2, 'alpha': .4})
    a.set_title(c); a.set_yticks([])
plt.suptitle('Boxplot audit outlier (data mentah)', y=1.02)
plt.tight_layout(); plt.show()

## 1.8 Keputusan pembersihan
Aturan dosen: *hindari mengubah/menghapus data asli tanpa alasan yang jelas.* Keputusan kita:

| Temuan | Keputusan | Alasan |
|---|---|---|
| Tidak ada missing value | Tidak ada yang diubah | Data sudah lengkap |
| ID_Pesanan berulang (multi-item) | **Dipertahankan** | Bukan duplikat murni; tiap baris adalah item berbeda (harga berbeda) |
| Outlier harga, ongkir, waktu | **Dipertahankan** | Nilai nyata (barang mahal, jarak jauh, keterlambatan). Ditangani dengan metode robust di analisis |
| Ongkos kirim = 0 | **Dipertahankan** | Wajar (gratis ongkir) |
| Waktu pengiriman = 0 hari | **Dipertahankan tetapi diberi penanda** | Kurang wajar, jumlahnya sangat sedikit; dicatat sebagai keterbatasan |

Jadi `df` (data analisis) isinya sama dengan data mentah, ditambah kolom penanda. Tidak ada baris yang dihapus.

In [ ]:
df = df_raw.copy()

# Kolom penanda (flag) — tidak menghapus apa pun
df['flag_item_ganda']   = df['ID_Pesanan'].duplicated(keep=False)
df['flag_waktu_nol']    = df['Waktu_Pengiriman_Hari'] == 0
df['flag_ongkir_nol']   = df['Ongkos_Kirim'] == 0

# Kelompok skor: tinggi (4-5) dan rendah (1-3), dipakai untuk ringkasan deskriptif
df['Kelompok_Ulasan'] = np.where(df['Skor_Kepuasan'] >= 4, 'Tinggi (4-5)', 'Rendah (1-3)')

# Versi pendukung: satu baris per pesanan (harga & ongkir dijumlah per pesanan;
# waktu & skor diambil dari baris pertama); disimpan sebagai data pendukung
df_per_pesanan = (df_raw.groupby('ID_Pesanan', as_index=False)
                  .agg(Harga_Produk=('Harga_Produk', 'sum'),
                       Ongkos_Kirim=('Ongkos_Kirim', 'sum'),
                       Waktu_Pengiriman_Hari=('Waktu_Pengiriman_Hari', 'first'),
                       Skor_Kepuasan=('Skor_Kepuasan', 'first')))

print('df (item pesanan)      :', df.shape)
print('df_per_pesanan         :', df_per_pesanan.shape)
df['Kelompok_Ulasan'].value_counts().to_frame('Jumlah')

## 1.9 Ringkasan audit

In [ ]:
ringkasan = pd.DataFrame({
    'Item': ['Jumlah baris (item pesanan)', 'Jumlah ID_Pesanan unik', 'Jumlah variabel (kolom)',
             'Missing value (total)', 'Baris duplikat murni', 'Baris dengan ID_Pesanan berulang',
             'Waktu pengiriman = 0 hari', 'Ongkos kirim = 0',
             'Skor di luar rentang 1-5'],
    'Nilai': [len(df_raw), df_raw['ID_Pesanan'].nunique(), df_raw.shape[1],
              int(df_raw.isna().sum().sum()), int(df_raw.duplicated().sum()),
              int(df_raw['ID_Pesanan'].duplicated().sum()),
              int(df['flag_waktu_nol'].sum()), int(df['flag_ongkir_nol'].sum()),
              int((~df_raw['Skor_Kepuasan'].isin([1,2,3,4,5])).sum())]})
ringkasan

In [ ]:
ringkasan.to_csv(f'{HASIL}/tabel_ringkasan_audit.csv', index=False)
audit_outlier.to_csv(f'{HASIL}/tabel_audit_outlier.csv')
K['audit'] = dict(zip(ringkasan['Item'], [int(v) for v in ringkasan['Nilai']]))
K['audit_outlier'] = {i: {c: float(audit_outlier.loc[i, c]) for c in audit_outlier.columns} for i in audit_outlier.index}
print('Tahap 1 selesai: tabel audit tersimpan di folder hasil/')

# TAHAP 2 — Statistika Deskriptif dan Visualisasi

**Catatan:** tahap ini hanya **menggambarkan** data (ringkasan angka, tabel frekuensi, dan grafik). Tidak ada uji hipotesis, pendugaan parameter populasi, ataupun model prediksi.

## A. Elemen data dan tipe variabel
Elemen data: banyak data (baris), banyak variabel (kolom), dan tipe tiap variabel.

In [ ]:
N = len(df)
K['n_item'] = N
K['n_pesanan'] = int(df['ID_Pesanan'].nunique())
K['n_kolom'] = int(df_raw.shape[1])

elemen = pd.DataFrame({
    'Elemen data': ['Banyak data (baris = item pesanan)', 'Banyak pesanan unik', 'Banyak kolom pada data utama',
                    'Banyak variabel yang dianalisis', 'Jumlah variabel numerik', 'Jumlah variabel pengenal (ID)'],
    'Nilai': [f'{N:,}', f"{K['n_pesanan']:,}", df_raw.shape[1], len(VAR_NUM), len(VAR_NUM), 1]})
elemen.to_csv(f'{HASIL}/tabel_elemen_data.csv', index=False)
elemen

In [ ]:
tipe = pd.DataFrame({
    'Variabel': ['ID_Pesanan', 'Harga_Produk', 'Ongkos_Kirim', 'Waktu_Pengiriman_Hari', 'Skor_Kepuasan'],
    'Jenis data': ['Kualitatif (pengenal)', 'Kuantitatif kontinu', 'Kuantitatif kontinu', 'Kuantitatif diskrit', 'Kuantitatif diskrit'],
    'Skala pengukuran': ['Nominal', 'Rasio', 'Rasio', 'Rasio', 'Ordinal'],
    'Satuan': ['-', 'BRL', 'BRL', 'hari', 'skor 1-5']})
tipe.to_csv(f'{HASIL}/tabel_tipe_variabel.csv', index=False)
tipe

Skor kepuasan berskala **ordinal** (urutan bermakna, jarak antar skor belum tentu sama). Rata-rata dan simpangan baku tetap dihitung karena lazim dipakai pada skala 1–5, tetapi **median dan modus** lebih tepat sebagai ukuran pemusatan untuk variabel ini.

## B. Tabel resume statistika deskriptif
Dihitung **17 ukuran**: pemusatan (mean, median, modus), penyebaran (simpangan baku, varians, galat baku, rentang, kuartil, IQR, koefisien variasi), bentuk sebaran (skewness, kurtosis), serta count, jumlah, minimum, dan maksimum.

In [ ]:
def deskriptif(s):
    q1, q2, q3 = s.quantile([.25, .5, .75])
    return pd.Series({
        'Count': s.count(), 'Jumlah': s.sum(), 'Mean': s.mean(), 'Galat baku (SE)': s.sem(),
        'Median (Q2)': q2, 'Modus': s.mode().iloc[0], 'Simpangan baku': s.std(), 'Varians': s.var(),
        'Min': s.min(), 'Q1': q1, 'Q3': q3, 'Maks': s.max(), 'Rentang (range)': s.max() - s.min(),
        'IQR': q3 - q1, 'Skewness': s.skew(), 'Kurtosis (excess)': s.kurt(),
        'Koef. variasi (%)': s.std() / s.mean() * 100})

tabel_desk = df[VAR_NUM].apply(deskriptif).T
tabel_desk.to_csv(f'{HASIL}/tabel_deskriptif.csv')
for v in VAR_NUM:
    K[f'desk_{v}'] = {c: float(tabel_desk.loc[v, c]) for c in tabel_desk.columns}
tabel_desk

**Cara membaca:** bila *mean* lebih besar dari *median* dan *skewness* positif, sebaran menjulur ke kanan (sebagian kecil nilai sangat besar menarik rata-rata ke atas). Bila skewness negatif, sebaran menjulur ke kiri. *Kurtosis (excess)* > 0 berarti ekor lebih berat dibanding sebaran normal. Koefisien variasi menunjukkan besar penyebaran relatif terhadap rata-rata.

## C. Distribusi frekuensi

In [ ]:
# Distribusi frekuensi skor kepuasan
frek_skor = df['Skor_Kepuasan'].value_counts().sort_index().to_frame('Frekuensi')
frek_skor['Persen (%)'] = frek_skor['Frekuensi'] / N * 100
frek_skor['Persen kumulatif (%)'] = frek_skor['Persen (%)'].cumsum()
frek_skor.to_csv(f'{HASIL}/tabel_frekuensi_skor.csv')
K['frek_skor'] = {int(k): float(v) for k, v in frek_skor['Persen (%)'].items()}
frek_skor

In [ ]:
# Distribusi frekuensi waktu pengiriman (dikelompokkan)
df['Kategori_Waktu'] = pd.cut(df['Waktu_Pengiriman_Hari'], bins=[-1, 7, 14, 21, np.inf],
                              labels=['0-7 hari', '8-14 hari', '15-21 hari', '> 21 hari'])
frek_waktu = df['Kategori_Waktu'].value_counts().sort_index().to_frame('Frekuensi')
frek_waktu['Persen (%)'] = frek_waktu['Frekuensi'] / N * 100
frek_waktu['Persen kumulatif (%)'] = frek_waktu['Persen (%)'].cumsum()
frek_waktu.to_csv(f'{HASIL}/tabel_frekuensi_waktu.csv')
K['frek_waktu'] = {str(k): float(v) for k, v in frek_waktu['Persen (%)'].items()}
frek_waktu

## D. Visualisasi tiap variabel
Setiap variabel divisualisasikan dengan histogram (atau diagram batang untuk skor) dan boxplot.

In [ ]:
INFO = {'Harga_Produk': ('Harga Produk (BRL)', 'gambar_01_harga_produk'),
        'Ongkos_Kirim': ('Ongkos Kirim (BRL)', 'gambar_02_ongkos_kirim'),
        'Waktu_Pengiriman_Hari': ('Waktu Pengiriman (hari)', 'gambar_03_waktu_pengiriman')}

def panel(v):
    nama, berkas = INFO[v]
    s = df[v]; batas = s.quantile(.99)
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.8), gridspec_kw={'width_ratios': [2, 1]})
    bins = np.arange(-0.5, batas + 1.5, 1) if v == 'Waktu_Pengiriman_Hari' else 50   # data hari = bilangan bulat
    ax[0].hist(s[s <= batas], bins=bins, color=BIRU, edgecolor='white', linewidth=.4)
    ax[0].axvline(s.mean(), color=ORANYE, lw=2, label=f'Mean = {s.mean():.2f}')
    ax[0].axvline(s.median(), color=INK, lw=1.6, ls='--', label=f'Median = {s.median():.2f}')
    ax[0].set_title(f'Histogram {nama}'); ax[0].set_xlabel(nama + ' (dipotong di persentil 99)')
    ax[0].set_ylabel('Frekuensi'); ax[0].legend(frameon=False, fontsize=9)
    bp = ax[1].boxplot(s, vert=True, patch_artist=True, widths=.5,
                       flierprops=dict(marker='o', markersize=2, markerfacecolor=NETRAL, markeredgecolor=NETRAL),
                       medianprops=dict(color=INK, lw=1.8))
    bp['boxes'][0].set(facecolor='#cfe0f7', edgecolor=BIRU)
    ax[1].set_xticks([1]); ax[1].set_xticklabels([nama]); ax[1].set_title(f'Boxplot {nama}'); ax[1].set_ylabel(nama)
    plt.tight_layout(); simpan(fig, berkas)

panel('Harga_Produk')

In [ ]:
panel('Ongkos_Kirim')

In [ ]:
panel('Waktu_Pengiriman_Hari')

In [ ]:
# Skor kepuasan: diagram batang + boxplot
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8), gridspec_kw={'width_ratios': [2, 1]})
ax[0].bar(frek_skor.index, frek_skor['Frekuensi'], color=BIRU, width=.7)
for x, f, p in zip(frek_skor.index, frek_skor['Frekuensi'], frek_skor['Persen (%)']):
    ax[0].text(x, f, f'{p:.1f}%', ha='center', va='bottom', fontsize=9, color=MUTED)
ax[0].set_title('Diagram Batang Skor Kepuasan'); ax[0].set_xlabel('Skor kepuasan (1-5)'); ax[0].set_ylabel('Frekuensi')
ax[0].set_ylim(0, frek_skor['Frekuensi'].max() * 1.12)
bp = ax[1].boxplot(df['Skor_Kepuasan'], vert=True, patch_artist=True, widths=.5, medianprops=dict(color=INK, lw=1.8),
                   flierprops=dict(marker='o', markersize=3, markerfacecolor=NETRAL, markeredgecolor=NETRAL))
bp['boxes'][0].set(facecolor='#cfe0f7', edgecolor=BIRU)
ax[1].set_xticks([1]); ax[1].set_xticklabels(['Skor Kepuasan']); ax[1].set_title('Boxplot Skor Kepuasan'); ax[1].set_ylabel('Skor')
plt.tight_layout(); simpan(fig, 'gambar_04_skor_kepuasan')

In [ ]:
# Diagram batang kategori waktu pengiriman
fig, a = plt.subplots(figsize=(6.5, 3.8))
a.bar(frek_waktu.index.astype(str), frek_waktu['Frekuensi'], color=BIRU, width=.65)
for i, (f, p) in enumerate(zip(frek_waktu['Frekuensi'], frek_waktu['Persen (%)'])):
    a.text(i, f, f'{p:.1f}%', ha='center', va='bottom', fontsize=9, color=MUTED)
a.set_title('Banyak Item Pesanan menurut Kategori Waktu Pengiriman'); a.set_xlabel('Kategori waktu pengiriman'); a.set_ylabel('Frekuensi')
a.set_ylim(0, frek_waktu['Frekuensi'].max() * 1.12)
plt.tight_layout(); simpan(fig, 'gambar_05_kategori_waktu')

## E. Ringkasan menurut kelompok skor
Skor 4–5 dikelompokkan sebagai *Tinggi* dan skor 1–3 sebagai *Rendah*, lalu tiap variabel diringkas dengan ukuran deskriptif.

In [ ]:
desk_kel = df.groupby('Kelompok_Ulasan')[VAR_X].agg(['count', 'mean', 'median', 'std'])
desk_kel.to_csv(f'{HASIL}/tabel_deskriptif_kelompok.csv')
K['kelompok'] = {g: {v: {m: float(desk_kel.loc[g, (v, m)]) for m in ['count', 'mean', 'median', 'std']} for v in VAR_X} for g in desk_kel.index}
desk_kel

## F. Ringkasan temuan dan penyimpanan hasil

In [ ]:
print('=== RINGKASAN TEMUAN DESKRIPTIF ===')
print(f'Banyak data: {N:,} item pesanan ({K["n_pesanan"]:,} pesanan unik), {len(VAR_NUM)} variabel dianalisis')
for v in VAR_NUM:
    d = K[f'desk_{v}']
    print(f"{v:24s} mean={d['Mean']:.2f}  median={d['Median (Q2)']:.2f}  modus={d['Modus']:.2f}  SD={d['Simpangan baku']:.2f}  skew={d['Skewness']:.2f}")
print('Persentase skor:', {k: round(v, 1) for k, v in K['frek_skor'].items()})
print('Persentase kategori waktu:', {k: round(v, 1) for k, v in K['frek_waktu'].items()})
with open(f'{HASIL}/angka_kunci.json', 'w') as f:
    json.dump(K, f, indent=1, ensure_ascii=False)

In [ ]:
# Kemas semua hasil menjadi satu ZIP lalu unduh
import shutil
shutil.make_archive('hasil_analisis', 'zip', HASIL)
try:
    from google.colab import files
    files.download('hasil_analisis.zip')
except ImportError:
    print('Tersimpan: hasil_analisis.zip')